# AI-Based Audio Classification & Edge Optimization

**Goal:** build a full pipeline that takes raw audio clips, extracts MFCC features, trains a CNN classifier across 5 sound classes (alarm and non-alarm), evaluates it, then optimizes it for edge deployment via post-training quantisation and TFLite export.

**Pipeline stages:**
1. Load and organize the audio dataset
2. Feature extraction (MFCC spectrograms)
3. Train/validation/test split
4. CNN model definition and training
5. Evaluation (accuracy, confusion matrix, per-class report)
6. Post-training quantisation (dynamic-range and full-integer)
7. TFLite export and edge-side accuracy comparison
8. Model size comparison (float vs quantised)

**Dataset layout expected:**
```
data/
  fire_alarm/
    clip_001.wav
    ...
  ambulance_siren/
    ...
  car_horn/
    ...
  dog_bark/
    ...
  background_noise/
    ...
```
`fire_alarm` and `ambulance_siren` are the two alarming classes this project cares about; the rest are non-alarm sounds the model must also recognise so it doesn't false-alarm on everyday noise.


In [ ]:
# --- Imports ---
import os
import glob
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


## 1. Configuration

Set the dataset path and class names here. `SAMPLE_RATE`, `DURATION`, and `N_MFCC` control how audio is loaded and how big each feature map is.

In [ ]:
DATA_DIR = "data"                 # folder containing one subfolder per class
CLASSES = sorted(os.listdir(DATA_DIR)) if os.path.isdir(DATA_DIR) else [
    "fire_alarm", "ambulance_siren", "car_horn", "dog_bark", "background_noise",
]  # used if DATA_DIR isn't populated yet — once your data/ folder exists, os.listdir(DATA_DIR) picks these up automatically

ALARM_CLASSES = {"fire_alarm", "ambulance_siren"}  # the classes that should count as an "alarm" detection

SAMPLE_RATE = 22050
DURATION = 4.0            # seconds — clips are padded/trimmed to this length
N_MFCC = 40                # number of MFCC coefficients
MAX_FRAMES = int(np.ceil(SAMPLE_RATE * DURATION / 512))  # hop_length=512

print(f"Classes ({len(CLASSES)}): {CLASSES}")


## 2. Feature extraction

Each audio clip is loaded, padded/trimmed to a fixed duration, and converted to an MFCC spectrogram. Fixed-size feature maps are needed so every sample has the same input shape for the CNN.

In [ ]:
def load_and_extract_mfcc(file_path, sr=SAMPLE_RATE, duration=DURATION, n_mfcc=N_MFCC, max_frames=MAX_FRAMES):
    """Load a wav file and return a fixed-size MFCC feature map."""
    audio, _ = librosa.load(file_path, sr=sr)

    target_len = int(sr * duration)
    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)

    # Pad or trim the time axis so every sample has identical shape
    if mfcc.shape[1] < max_frames:
        pad_width = max_frames - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)), mode="constant")
    else:
        mfcc = mfcc[:, :max_frames]

    return mfcc.astype(np.float32)


def build_dataset(data_dir, classes):
    X, y = [], []
    for label_idx, class_name in enumerate(classes):
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        for file_path in glob.glob(os.path.join(class_dir, "*.wav")):
            try:
                features = load_and_extract_mfcc(file_path)
                X.append(features)
                y.append(label_idx)
            except Exception as e:
                print(f"Skipping {file_path}: {e}")
    return np.array(X), np.array(y)


if os.path.isdir(DATA_DIR) and len(os.listdir(DATA_DIR)) > 0:
    X, y = build_dataset(DATA_DIR, CLASSES)
    print(f"Loaded {len(X)} samples across {len(CLASSES)} classes")
else:
    print("DATA_DIR not populated — point DATA_DIR at your dataset before running training.")


## 3. Train / validation / test split

An 70/15/15 split, stratified by class so the class balance is preserved across splits. Class weights are computed to reduce the effect of any class imbalance during training.

In [ ]:
# Add a channel dimension for Conv2D: (samples, n_mfcc, frames, 1)
X = X[..., np.newaxis]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=y_temp
)

print(f"Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}")

class_weights_arr = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_train), y=y_train
)
class_weights = dict(enumerate(class_weights_arr))
print("Class weights:", class_weights)


## 4. CNN model

A compact CNN: three conv blocks (Conv2D + BatchNorm + MaxPool), then a dense classification head. Kept small deliberately, since the end goal is edge deployment where model size and inference latency matter as much as accuracy.

In [ ]:
def build_model(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),

        tf.keras.layers.Conv2D(16, (3, 3), padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),

        tf.keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),

        tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),

        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])
    return model


model = build_model(input_shape=X_train.shape[1:], num_classes=len(CLASSES))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


## 5. Training

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=2,
)


## 6. Evaluation

Accuracy plus a full classification report and confusion matrix on the held-out test set — this is the baseline (float32) model's performance before any quantisation.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Float32 model — test accuracy: {test_acc:.4f}")

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print(classification_report(y_test, y_pred, target_names=CLASSES))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


## 7. Post-training quantisation and TFLite export

Two quantisation strategies:
- **Dynamic-range quantisation** — weights quantised to int8, activations computed in float at inference time. Simple, no calibration data needed.
- **Full integer quantisation** — both weights and activations quantised to int8, using a representative dataset for calibration. Produces the smallest, fastest model, best suited for microcontroller-class edge hardware.


In [ ]:
model.save("audio_cnn_float32.keras")

# --- Dynamic-range quantisation ---
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_dynamic = converter.convert()
with open("audio_cnn_dynamic_quant.tflite", "wb") as f:
    f.write(tflite_dynamic)

# --- Full integer quantisation (with representative dataset) ---
def representative_dataset():
    for sample in X_train[:200]:
        yield [np.expand_dims(sample, axis=0)]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.int8
converter_int8.inference_output_type = tf.int8
tflite_int8 = converter_int8.convert()
with open("audio_cnn_int8_quant.tflite", "wb") as f:
    f.write(tflite_int8)

print("Saved: audio_cnn_float32.keras, audio_cnn_dynamic_quant.tflite, audio_cnn_int8_quant.tflite")


## 8. Evaluating the quantised models

Run the TFLite interpreter directly (this is what an edge device would actually execute) and compare accuracy against the float32 baseline.

In [ ]:
def evaluate_tflite(tflite_path, X_eval, y_eval, is_int8=False):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    correct = 0
    for i in range(len(X_eval)):
        sample = X_eval[i:i+1]
        if is_int8:
            scale, zero_point = input_details["quantization"]
            sample = (sample / scale + zero_point).astype(np.int8)
        else:
            sample = sample.astype(np.float32)

        interpreter.set_tensor(input_details["index"], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details["index"])
        pred = np.argmax(output)
        if pred == y_eval[i]:
            correct += 1

    return correct / len(X_eval)


acc_dynamic = evaluate_tflite("audio_cnn_dynamic_quant.tflite", X_test, y_test, is_int8=False)
acc_int8 = evaluate_tflite("audio_cnn_int8_quant.tflite", X_test, y_test, is_int8=True)

print(f"Float32 accuracy:          {test_acc:.4f}")
print(f"Dynamic-range TFLite acc:  {acc_dynamic:.4f}  (drop: {test_acc - acc_dynamic:.4f})")
print(f"Full-int8 TFLite acc:      {acc_int8:.4f}  (drop: {test_acc - acc_int8:.4f})")

assert (test_acc - acc_int8) < 0.02, "Quantised accuracy drop exceeded the 2% target"


## 9. Model size comparison — the actual point of edge optimisation

In [ ]:
def file_size_kb(path):
    return os.path.getsize(path) / 1024

sizes = {
    "Float32 Keras model": "audio_cnn_float32.keras",
    "Dynamic-range TFLite": "audio_cnn_dynamic_quant.tflite",
    "Full-int8 TFLite": "audio_cnn_int8_quant.tflite",
}

for name, path in sizes.items():
    print(f"{name}: {file_size_kb(path):.1f} KB")


## Summary

- Built a full pipeline from raw `.wav` clips to a deployable edge model: MFCC feature extraction → CNN training → evaluation → post-training quantisation → TFLite export.
- Compared dynamic-range and full-integer (int8) quantisation against the float32 baseline.
- Target: keep the accuracy drop from quantisation under 2%, while cutting model size substantially — the standard tradeoff for deploying a classifier on constrained edge hardware (microcontrollers, mobile, embedded boards) rather than a server.
